In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab11.ipynb")

# Lab 11: U.S. Weather Data — Least Squares and Residuals

Use real **2024 daily weather observations from Boston Logan International Airport, Massachusetts**, to examine the relationship between dew point and mean air temperature. Compare proposed prediction rules, fit least-squares models, and assess whether wind speed provides useful additional information.

The lab applies numerical optimization, multiple regression, error measures, and residual diagnostics from Week11 Lectures 20–21.

### Libraries and Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

from scipy.optimize import minimize

## Dataset and Source

The supplied `environment_weather_2024.csv` contains **366 daily observations**, January 1–December 31, 2024. These are measured weather summaries from NOAA's **Global Summary of the Day (GSOD)** dataset; no observations were simulated.

- **Station:** Boston Logan International Airport, MA, USA; NOAA station ID `72509014739`.
- **Original data:** [NOAA 2024 station CSV](https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/2024/72509014739.csv).
- **Variable definitions:** [NOAA GSOD documentation](https://www.ncei.noaa.gov/data/global-summary-of-the-day/doc/readme.txt).
- **Preparation:** retain `DATE`, `DEWP`, `WDSP`, and `TEMP`; convert Fahrenheit to Celsius and knots to meters per second. Replace NOAA missing-value codes before excluding incomplete records. All 366 days have valid values for these three measurements, so no days were excluded. Sort chronologically and round converted values to six decimal places in the supplied CSV. No missing observations were imputed.

| Column | Description | Unit |
|---|---|---|
| `date` | Observation date | YYYY-MM-DD |
| `dewpoint_c` | Mean daily dew point | °C |
| `wind_m_s` | Mean daily wind speed | m/s |
| `temperature_c` | Mean daily air temperature | °C |

**Dew point** is the temperature at which air becomes saturated with water vapor. It describes atmospheric moisture; it is not the same measurement as air temperature. The models describe associations among weather measurements at one station during one year.

The student ZIP also includes NOAA's original CSV and a provenance README. The lab uses the prepared CSV and requires no live data downloads.

## Instructions

Keep the prepared CSV beside this notebook. Run cells in order and preserve the observation order. Use population standard deviations (`ddof=0`). Do not overwrite earlier answer variables. Data preparation and the numerical objective function are supplied; no student-defined functions are required.

Complete code, figures, and explanations in the designated cells. Public tests provide feedback; hidden tests also contribute to your grade. Discuss approaches with others, but submit your own work.

### Data Preparation

The following cell imports the required libraries, loads the NOAA-derived CSV, and prepares the arrays and summary statistics used in the lab. It also defines `dewpoint_mse(parameters)`, the MSE objective for Question 2, with parameters ordered as **slope, intercept**.

**Run this cell without changing it.**


In [ ]:
# Provided data preparation: run without changing.
air_quality = pd.read_csv('data/environment_weather_2024.csv')
dewpoint_values = air_quality['dewpoint_c'].to_numpy()
wind_values = air_quality['wind_m_s'].to_numpy()
temperature_values = air_quality['temperature_c'].to_numpy()
n_days = len(air_quality)
dewpoint_mean = np.mean(dewpoint_values)
dewpoint_sd = np.std(dewpoint_values)
temperature_mean = np.mean(temperature_values)
temperature_sd = np.std(temperature_values)
dewpoint_temperature_r = np.corrcoef(dewpoint_values, temperature_values)[0, 1]
air_quality.head()

In [ ]:
# Provided objective for Question 2. The two parameters are slope, intercept.
def dewpoint_mse(parameters):
    estimates = parameters[0] * dewpoint_values + parameters[1]
    residual_values = temperature_values - estimates
    return np.mean(residual_values ** 2)

---
<a id="outline"></a>
## Question Navigation

- [Question 1: Model Comparison](#q1)
- [Question 2: Numerical Optimization](#q2)
- [Question 3: Least-Squares Fit](#q3)
- [Question 4: Prediction Errors](#q4)
- [Question 5: Baseline Comparison](#q5)
- [Question 6: Multiple Regression](#q6)
- [Question 7: Residual Associations](#q7)
- [Question 8: Diagnostic Plots](#q8)
- [Question 9: Technical Interpretation](#q9)


<a id="q1"></a>
### Question 1: Model Comparison (10 points)

Two proposed models predict mean daily air temperature from mean daily dew point:

$$\text{Model A: }\widehat y=0.8x+8
\qquad
\text{Model B: }\widehat y=1.1x+4,$$

where $x$ is dew point in °C and $\widehat y$ is predicted mean air temperature in °C. Evaluate both models using all 366 observations.

- Calculate `candidate_a_predictions` and `candidate_b_predictions`.
- Calculate `candidate_a_residuals` and `candidate_b_residuals` as **observed minus predicted** temperature.
- Compute `candidate_a_mse` and `candidate_b_mse`.
- Assign `better_candidate` to `'A'` or `'B'`, selecting the model with the lower MSE.

Preserve the original observation order. Use the fixed coefficients above for this comparison.


In [ ]:
candidate_a_predictions = ...
candidate_b_predictions = ...
candidate_a_residuals = ...
candidate_b_residuals = ...
candidate_a_mse = ...
candidate_b_mse = ...
better_candidate = ...

In [ ]:
grader.check("q1")

[Return to outline](#outline)

<a id="q2"></a>
### Question 2: Numerical Optimization (10 points)

Determine the slope and intercept that minimize mean squared prediction error for the dew-point-only model:

$$\widehat y=mx+b.$$

Use the supplied `dewpoint_mse` function with SciPy's `minimize`. Set the initial parameter values to `[0, 0]` and use `method='Powell'`.

1. Store the optimization result in `optimization_result`.
2. Extract the slope and intercept from `optimization_result.x` as `optimized_slope` and `optimized_intercept`.
3. Calculate `optimized_predictions` and their mean squared error, `optimized_mse`.
4. Calculate `mse_gain_over_candidates`: the lower of the two MSE values from Question 1 minus `optimized_mse`.

The supplied objective accepts parameters in the order **slope, intercept**. The optimizer's `.x` attribute contains the fitted parameters, and `.success` indicates convergence.


In [ ]:
optimization_result = ...
optimized_slope = ...
optimized_intercept = ...
optimized_predictions = ...
optimized_mse = ...
mse_gain_over_candidates = ...

In [ ]:
grader.check("q2")

[Return to outline](#outline)

<a id="q3"></a>
### Question 3: Least-Squares Fit (11 points)

Estimate the dew-point-only model analytically:

$$\widehat y=mx+b,\qquad m=r\frac{SD(y)}{SD(x)},\qquad b=\overline y-m\overline x.$$

Use the supplied correlation, means, and population standard deviations.

- Store the coefficients in `simple_slope` and `simple_intercept`.
- Calculate `simple_predictions` for every observed dew point.
- Calculate `temperature_at_10`, the predicted mean air temperature when mean dew point is 10°C.

This fit and the numerical fit in Question 2 minimize the same objective. Preserve both results for comparison.


In [ ]:
simple_slope = ...
simple_intercept = ...
simple_predictions = ...
temperature_at_10 = ...

In [ ]:
grader.check("q3")

[Return to outline](#outline)

<a id="q4"></a>
### Question 4: Prediction Errors (12 points)

Evaluate the analytical model from Question 3.

1. Calculate `simple_residuals` as observed temperature minus `simple_predictions`.
2. Calculate `simple_mae`, `simple_mse`, and `simple_rmse` using all observations.
3. Find the row position with the largest **absolute** residual and store it in `largest_error_position`.
4. Store that observation's date in `largest_error_date`, using the original date string from the CSV.

If several observations tie, select the first in the original row order. A negative residual indicates that the model predicted a temperature above the observed value.


In [ ]:
simple_residuals = ...
simple_mae = ...
simple_mse = ...
simple_rmse = ...
largest_error_position = ...
largest_error_date = ...

In [ ]:
grader.check("q4")

[Return to outline](#outline)

<a id="q5"></a>
### Question 5: Baseline Comparison (12 points)

A mean-only baseline predicts the observed mean air temperature for every day. Evaluate whether using dew point improves the fit relative to this baseline.

- Store the baseline's constant prediction in `baseline_prediction`.
- Calculate its actual-minus-predicted residuals as `baseline_residuals`, then compute `baseline_rmse`.
- Calculate `rmse_reduction_pct` using

$$100\frac{RMSE_{\mathrm{baseline}}-RMSE_{\mathrm{simple}}}{RMSE_{\mathrm{baseline}}}.$$

- Create `rmse_comparison`, a pandas Series containing the two RMSE values, with index labels `'baseline'` and `'simple'` in that order.
- Store the index label with the smaller RMSE in `better_model`.

Both models are evaluated on the observations used to fit them.


In [ ]:
baseline_prediction = ...
baseline_residuals = ...
baseline_rmse = ...
rmse_reduction_pct = ...
rmse_comparison = ...
better_model = ...

In [ ]:
grader.check("q5")

[Return to outline](#outline)

<a id="q6"></a>
### Question 6: Multiple Regression (12 points)

Dew point alone may not account for all systematic variation in temperature. Consider a model that also includes wind speed:

$$\widehat y=a\,\mathrm{dewpoint}+b\,\mathrm{wind}+c.$$

The supplied fitting code uses `np.linalg.lstsq` to minimize squared prediction errors. Run the provided lines without changing them; deriving the solver is not required.

1. Extract the three fitted parameters from `multiple_coefficients`, in order, as `dewpoint_coefficient`, `wind_coefficient`, and `multiple_intercept`.
2. Calculate `multiple_predictions` using the fitted equation. Use short intermediate steps where helpful.
3. Calculate `multiple_rmse` from the observed and predicted temperatures.
4. Calculate `multiple_rmse_gain` as `simple_rmse - multiple_rmse`.

Both regression models must use the same observations so their fitting errors are comparable.


In [ ]:
# Provided fitting code: do not change.
design_matrix = np.column_stack([dewpoint_values, wind_values, np.ones(n_days)])
multiple_coefficients = np.linalg.lstsq(design_matrix, temperature_values, rcond=None)[0]

dewpoint_coefficient = ...
wind_coefficient = ...
multiple_intercept = ...
multiple_predictions = ...
# You may use additional short steps.
multiple_rmse = ...
multiple_rmse_gain = ...

In [ ]:
grader.check("q6")

[Return to outline](#outline)

<a id="q7"></a>
### Question 7: Residual Associations (12 points)

Examine whether the residuals retain a linear association with wind speed after wind is included in the model.

- Calculate `multiple_residuals` as observed minus predicted temperature for the model in Question 6.
- Calculate their mean as `multiple_residual_mean`.
- Convert wind speed, both residual arrays, and the multiple-model predictions to standard units using population standard deviations.
- Calculate each Pearson correlation as the **mean of the elementwise products of the corresponding standard-unit arrays**. **Do not use `np.corrcoef`**.

| Variable | Quantities to correlate |
|---|---|
| `simple_wind_r` | `simple_residuals` and `wind_values` |
| `multiple_wind_r` | `multiple_residuals` and `wind_values` |
| `multiple_fitted_r` | `multiple_residuals` and `multiple_predictions` |

Preserve the original observation order. These diagnostics describe the observations used to fit the models.

In [ ]:
multiple_residuals = ...
multiple_residual_mean = ...

wind_su = ...

simple_residuals_su = ...
simple_residuals_su = ...

multiple_residuals_su = ...
multiple_residuals_su = ...

multiple_predictions_su = ...
multiple_predictions_su = ...

simple_wind_r = ...
multiple_wind_r = ...
multiple_fitted_r = ...

In [ ]:
grader.check("q7")

[Return to outline](#outline)

<!-- BEGIN QUESTION -->

<a id="q8"></a>
### Question 8: Diagnostic Plots (12 points)

Prepare one figure with three panels to support a technical comparison of the models.

| Panel | Horizontal axis | Vertical axis | Required reference |
|---|---|---|---|
| 1 | Mean dew point (°C) | Observed mean air temperature (°C) | Dew-point-only fitted line |
| 2 | Mean wind speed (m/s) | Dew-point-only residuals (°C) | Horizontal line at zero |
| 3 | Mean wind speed (m/s) | Dew-point-and-wind residuals (°C) | Horizontal line at zero |

Use scatter plots for observations and residuals. Draw the fitted line using dew points sorted in ascending order, without modifying the original arrays. Use identical horizontal and vertical axis limits for the two residual panels to allow a fair comparison.

Include informative panel titles, labeled axes with units, and a legend distinguishing observations from the fitted line in panel 1. Make all three panels readable.

**Grading Rubric (12 points):** observed data and fitted line in panel 1 (5 points); correct residual data, zero lines, and comparable scales in panels 2–3 (5 points); clear labels, legend, and layout (2 points).


In [ ]:
# Create the three-panel figure here.


<!-- END QUESTION -->

[Return to outline](#outline)

<!-- BEGIN QUESTION -->

<a id="q9"></a>
### Question 9: Technical Interpretation (9 points)

Use the computed errors, correlations, and plots to decide whether adding wind speed materially improves the temperature model. Provide three numbered responses supported by evidence.

**1. Residual pattern (3 points).**
Describe the relationship between wind speed and the dew-point-only residuals in panel 2. Report `simple_wind_r`. Does the evidence indicate a strong remaining linear association, or a weak one? Explain whether the plot supports a consistent tendency to overpredict or underpredict at higher wind speeds. Use the actual-minus-predicted definition of a residual in your explanation.

**2. Practical Value of the Added Predictor (3 points).** Report `simple_rmse`, `multiple_rmse`, and `multiple_rmse_gain` in °C to four decimal places. (1 point). Compare panels 2 and 3 and report `multiple_wind_r`. Explain whether adding wind produces a substantial improvement or only a small change. (1 point). Explain why a lower fitting RMSE alone does not establish better prediction accuracy on future days, and state how you would evaluate this. (1 point)

**3. Residual mean and prediction accuracy (3 points).**
Report `multiple_residual_mean` and `multiple_rmse`. Explain how a mean residual close to zero can coexist with a nonzero RMSE. Refer to the spread of points around zero in panel 3 to explain why the residual mean alone is insufficient to judge prediction accuracy.

Interpret the evidence rather than listing values alone. A near-zero value may be reported in scientific notation or described as approximately zero.


*Write your interpretation here.*

<!-- END QUESTION -->

[Return to outline](#outline)

## Congratulations!

You have completed Lab 11! You used real U.S. weather data to fit regression models, evaluate prediction errors, and interpret residual patterns.

- Restart the kernel and run all cells in order.
- Review your answers, plots, and written interpretations, then save and submit the completed `zip` file on Canvas.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)